In [ ]:
# ==========================================
# 🛡️ GUARDIÃO DE VERSÃO 🛡️
# ==========================================
VERSAO_LOCAL = "2026-09-24T00:06:03Z"
NOME_ARQUIVO = "Transcribe.ipynb"

import urllib.request, json, re
URL = f"https://raw.githubusercontent.com/gabrieltrovao12/Medhelp/main/scripts/colab/{NOME_ARQUIVO}"
print("🔍 Checando se há atualizações...")
try:
    with urllib.request.urlopen(URL) as req:
        remoto = req.read().decode('utf-8')
VERSAO_LOCAL = "2026-09-24T00:06:03Z"
    if match:
        versao_remota = match.group(1)
VERSAO_LOCAL = "2026-09-24T00:06:03Z"
            raise Exception("❌ NOTEBOOK DESATUALIZADO! Atualize a página (F5) para puxar o código novo.")
    print("✅ Guardião: Versão sincronizada com o GitHub.")
except Exception as e:
    if "DESATUALIZADO" in str(e): raise e
    print(f"⚠️ Aviso do Guardião: Não foi possível checar a versão ({e})")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 1 — INSTALAÇÃO DE DEPENDÊNCIAS              ║
# ║  Execute apenas UMA VEZ por sessão do Colab.        ║
# ╚══════════════════════════════════════════════════════╝

print('Instalando pacotes Python...')
!pip install PyPDF2 pytesseract pdf2image Pillow requests -q

print('Instalando faster-whisper e dependências CUDA...')
!pip install faster-whisper nvidia-cublas-cu12 nvidia-cudnn-cu12 -q

# ── Injetar LD_LIBRARY_PATH para libs CUDA do pip ──
import os, site
_cuda_dirs = []
for sp in site.getsitepackages():
    for pkg in ['nvidia/cublas/lib', 'nvidia/cudnn/lib', 'nvidia_cublas_cu12']:
        d = os.path.join(sp, pkg)
        if os.path.isdir(d):
            _cuda_dirs.append(d)
for d in ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']:
    if os.path.isdir(d):
        _cuda_dirs.append(d)
if _cuda_dirs:
    existing = os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_LIBRARY_PATH'] = ':'.join(_cuda_dirs) + (':' + existing if existing else '')
    print(f'🔧 LD_LIBRARY_PATH atualizado com {len(_cuda_dirs)} diretório(s) CUDA.')

print('Instalando ffmpeg, Tesseract e Poppler...')
!sudo apt-get update -qq
!sudo apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-por poppler-utils -qq

print('\n✅ FOGUETE PRONTO — Dependências instaladas (v2.5 CUDA-FIX).')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 2 — MONTAGEM DO DRIVE E IMPORTS             ║
# ╚══════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount('/content/drive')

import os
import PyPDF2
import subprocess
import shutil
import time
from datetime import datetime
from pathlib import Path

print('✅ Drive montado e bibliotecas importadas.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado e bibliotecas importadas.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 3 — CONFIGURAÇÃO DO LOTE                                    ║
# ║                                                                      ║
# ║  Edite apenas esta célula para configurar suas aulas.               ║
# ║                                                                      ║
# ║  CAMPOS DE CADA AULA:                                                ║
# ║  • nome_saida      → "03 - Tema Da Aula" (sem extensão)            ║
# ║  • caminhos_audios → Lista de áudios (1 parte ou várias partes)     ║
# ║  • caminho_pdf     → Caminho do PDF de slides (ou None se não tiver)║
# ║  • prompt_whisper  → Termos médicos para guiar o Whisper            ║
# ╚══════════════════════════════════════════════════════════════════════╝

import re
import os
import subprocess
import sys
from pathlib import Path

# ──────────────────────────────────────────────
# PASTA DE SAÍDA NO DRIVE (onde os .txt vão)
# ──────────────────────────────────────────────
PASTA_SAIDA_DRIVE = "/content/drive/MyDrive/Logística - UNDB/Transcrições - UNDB/Transcricoes_Medicina - UNDB"

# ──────────────────────────────────────────────
# OPÇÕES AVANÇADAS DO WHISPER
# ──────────────────────────────────────────────
WHISPER_CONFIG = {
    "model":                      "large-v3",
    "language":                   "pt",
    "temperature":                0.0,
    "condition_on_previous_text": False,
}


# ──────────────────────────────────────────────
# LOTE DE AULAS
# ──────────────────────────────────────────────
PREFIXO_AUDIO = "/content/drive/MyDrive/Áudios aulas/"

aulas_para_processar = [

    # ── AULA 1 ──
    {
        "nome_saida": "LHM - Doenças Relacionadas Ao Trabalho",
        "caminhos_audios": [
            PREFIXO_AUDIO + "Conferência - Doenças relacionadas ao trabalho.m4a",
        ],
        "caminho_pdf": "/content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P5 - Saúde do Trabalhador/NOÇÕES BÁSICAS DE MEDICINA DO TRABALHO.pptx.pdf",
        "prompt_whisper": "Aula de Medicina: Medicina do trabalho, pneumoconiose, silicose, asbestose, mesotelioma, saturnismo, hidrargirismo, benzenismo, asma ocupacional, PAIR, LER, DORT, síndrome de burnout, nexo causal, CAT, ergonomia, toxicologia ocupacional, bissinose, beriliose, pneumonite de hipersensibilidade, radiação ionizante, leucopenia, anemia aplástica, chumbo, mercúrio, cromo, dermatite de contato ocupacional, surdez neurossensorial, neuropatia periférica, epicondilite, tenossinovite de De Quervain, síndrome do túnel do carpo, vibração de corpo inteiro, ruído ocupacional, poeiras inorgânicas, espirometria, audiometria, exame toxicológico, limite de tolerância, insalubridade",
    },

    # ── AULA 2 ──
    {
        "nome_saida": "LHM - Drogas De Abuso",
        "caminhos_audios": [
            PREFIXO_AUDIO + "Conferência - drogas de abuso - parte 01 20260527-141035.m4a",
            PREFIXO_AUDIO + "Conferência - drogas de abuso - parte 02 20260527-141653.m4a",
        ],
        "caminho_pdf": "/content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P2 - Drogas Ilícitas/AULA - Drogas Ilícitas.pdf",
        "prompt_whisper": "Aula de Medicina: Toxicologia, psiquiatria, dependência química, tolerância, síndrome de abstinência, intoxicação aguda, overdose, cocaína, crack, maconha, THC, canabidiol, anfetaminas, metanfetamina, MDMA, ecstasy, opioides, heroína, fentanil, morfina, naloxona, flumazenil, benzodiazepínicos, barbitúricos, LSD, alucinógenos, cetamina, álcool, delirium tremens, síndrome de Wernicke-Korsakoff, hepatopatia alcoólica, miose, midríase, depressão respiratória, taquicardia, hipertensão, agitação psicomotora, psicose tóxica, dopamina, sistema de recompensa",
    },
]

# ──────────────────────────────────────────────
# Instalação de dependências do sistema (OCR)
# ──────────────────────────────────────────────
subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', '-qq',
     'poppler-utils', 'tesseract-ocr', 'tesseract-ocr-por'],
    check=True
)
print('✅ Poppler e Tesseract (português) instalados.')

def _instalar_se_ausente(pacote_pip, import_name=None):
    import importlib
    nome = import_name or pacote_pip
    try:
        importlib.import_module(nome)
    except ImportError:
        print(f'[DEP] Instalando {pacote_pip}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pacote_pip, '-q'], check=True)

_instalar_se_ausente('pytesseract')
_instalar_se_ausente('pdf2image')
_instalar_se_ausente('Pillow', 'PIL')

# ──────────────────────────────────────────────
# Função que formata o nome final do arquivo
# ──────────────────────────────────────────────
def formatar_nome_saida(nome_saida: str) -> str:
    return f'{nome_saida.strip()} (Resumo)'

# ──────────────────────────────────────────────
# Validação de caminhos
# ──────────────────────────────────────────────
print('\nValidando configuração do lote...\n')
erros_encontrados = False

for i, aula in enumerate(aulas_para_processar):
    nome_final = formatar_nome_saida(aula["nome_saida"])
    print(f'[Aula {i+1}] {nome_final}')

    for j, caminho_audio in enumerate(aula["caminhos_audios"]):
        existe = os.path.exists(caminho_audio)
        status = '✅' if existe else '❌ ARQUIVO NÃO ENCONTRADO'
        print(f'  Áudio {j+1}: {status} → {caminho_audio}')
        if not existe:
            erros_encontrados = True

    if aula["caminho_pdf"]:
        existe_pdf = os.path.exists(aula["caminho_pdf"])
        status_pdf = '✅' if existe_pdf else '❌ ARQUIVO NÃO ENCONTRADO'
        print(f'  PDF:     {status_pdf} → {aula["caminho_pdf"]}')
        if not existe_pdf:
            erros_encontrados = True
    else:
        print('  PDF:     ⚠️  Nenhum PDF configurado para esta aula.')

    priming = aula.get("prompt_whisper", "").strip()
    if priming:
        print(f'  Priming: ✅ ({len(priming.split(","))} termos)')
    else:
        print('  Priming: ⚠️  Não definido — Whisper rodará sem priming.')

    print()

if erros_encontrados:
    print('⛔ ATENÇÃO: Corrija os caminhos acima antes de executar a Célula 4.')
else:
    print(f'✅ Tudo certo! {len(aulas_para_processar)} aula(s) configurada(s). Execute a Célula 4.')

✅ Poppler e Tesseract (português) instalados.

Validando configuração do lote...

[Aula 1] LHM - Doenças Relacionadas Ao Trabalho (Resumo)
  Áudio 1: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - Doenças relacionadas ao trabalho.m4a
  PDF:     ✅ → /content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P5 - Saúde do Trabalhador/NOÇÕES BÁSICAS DE MEDICINA DO TRABALHO.pptx.pdf
  Priming: ✅ (40 termos)

[Aula 2] LHM - Drogas De Abuso (Resumo)
  Áudio 1: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - drogas de abuso - parte 01 20260527-141035.m4a
  Áudio 2: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - drogas de abuso - parte 02 20260527-141653.m4a
  PDF:     ✅ → /content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P2 - Drogas Ilícitas/AULA - Drogas Ilícitas.pdf
  Priming: ✅ (40 termos)

✅ Tudo certo! 2 aula(s) configurada(s). Execute a Célula 4.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 4 — MOTOR DE EXECUÇÃO                                       ║
# ║  Não edite esta célula. Apenas execute-a.                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Imports principais ────────────────────────────────────────────────
import os
import re
import shutil
import time
import subprocess
import PyPDF2
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from datetime import datetime
from pathlib import Path

# ── Constantes internas ───────────────────────────────────────────────
DIR_TEMP            = "/content/_whisper_temp"
SEPARADOR           = "\n\n[--- PAUSA NA GRAVAÇÃO / CONTINUAÇÃO DA AULA ---]\n\n"
LIMITE_CHARS_PAGINA = 50


# ═════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ═════════════════════════════════════════════════════════════════════

def log(msg: str, nivel: str = 'INFO'):
    ts = datetime.now().strftime('%H:%M:%S')
    prefixos = {'INFO': '   ', 'OK': '✅ ', 'WARN': '⚠️ ', 'ERR': '❌ ', 'STEP': '▶️ ', 'OCR': '🔍 '}
    print(f"[{ts}] {prefixos.get(nivel, '')} {msg}")

def formatar_duracao(segundos: float) -> str:
    if segundos <= 0.0:
        return "Não foi possível determinar (ffprobe falhou)"
    h = int(segundos // 3600)
    m = int((segundos % 3600) // 60)
    s = int(segundos % 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def obter_duracao_audio_segundos(caminho_audio: str) -> float:
    """Retorna a duração em segundos usando ffprobe. Retorna 0.0 em caso de falha."""
    try:
        resultado = subprocess.run(
            [
                'ffprobe', '-v', 'error',
                '-show_entries', 'format=duration',
                '-of', 'default=noprint_wrappers=1:nokey=1',
                caminho_audio
            ],
            capture_output=True, text=True
        )
        return float(resultado.stdout.strip())
    except Exception as e:
        log(f'Não foi possível obter duração de {Path(caminho_audio).name}: {e}', 'WARN')
        return 0.0

def _garantir_ld_library_path():
    """Injeta caminhos de libs CUDA no LD_LIBRARY_PATH em runtime."""
    import site as _site
    cuda_dirs = []
    for sp in _site.getsitepackages():
        for pkg in ['nvidia/cublas/lib', 'nvidia/cudnn/lib', 'nvidia_cublas_cu12']:
            d = os.path.join(sp, pkg)
            if os.path.isdir(d):
                cuda_dirs.append(d)
    for d in ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']:
        if os.path.isdir(d):
            cuda_dirs.append(d)
    if cuda_dirs:
        existing = os.environ.get('LD_LIBRARY_PATH', '')
        new_paths = ':'.join(cuda_dirs)
        if new_paths not in existing:
            os.environ['LD_LIBRARY_PATH'] = new_paths + (':' + existing if existing else '')
            import ctypes
            for d in cuda_dirs:
                for lib_file in sorted(Path(d).glob('libcublas*.so*')):
                    try:
                        ctypes.cdll.LoadLibrary(str(lib_file))
                        break
                    except OSError:
                        continue
            log(f'LD_LIBRARY_PATH injetado com {len(cuda_dirs)} dir(s) CUDA.', 'INFO')

def transcrever_audio(caminho_audio, parte_num, total_partes, prompt, config):
    log(f'Transcrevendo parte {parte_num}/{total_partes}: {Path(caminho_audio).name}', 'STEP')
    if not os.path.exists(caminho_audio):
        raise FileNotFoundError(f'Áudio não encontrado: {caminho_audio}')
    
    from faster_whisper import WhisperModel
    global _modelo_whisper
    if '_modelo_whisper' not in globals():
        _garantir_ld_library_path()
        log('Carregando modelo faster-whisper na VRAM...', 'INFO')
        try:
            _modelo_whisper = WhisperModel(config['model'], device='cuda', compute_type='float16')
        except RuntimeError as e:
            log(f'CUDA falhou ({e}). Tentando com CPU (mais lento)...', 'WARN')
            _modelo_whisper = WhisperModel(config['model'], device='cpu', compute_type='int8')
    
    segmentos, info = _modelo_whisper.transcribe(
        caminho_audio,
        language=config['language'],
        temperature=config['temperature'],
        condition_on_previous_text=config['condition_on_previous_text'],
        initial_prompt=prompt if prompt and prompt.strip() else None
    )
    
    texto = ' '.join([s.text for s in segmentos]).strip()
    log(f'Parte {parte_num} transcrita. Caracteres: {len(texto):,}', 'OK')
    return texto

def ocr_pagina(imagem: Image.Image, num_pagina: int) -> str:
    """Aplica OCR em uma imagem de página usando Tesseract."""
    try:
        return pytesseract.image_to_string(imagem, lang='por').strip()
    except Exception as e:
        log(f'OCR falhou na página {num_pagina}: {e}', 'WARN')
        return ''

def _ler_texto_direto_pdf(caminho_pdf: str) -> list:
    """Lê o texto das páginas diretamente usando PyPDF2."""
    with open(caminho_pdf, 'rb') as f:
        leitor = PyPDF2.PdfReader(f)
        total_paginas = len(leitor.pages)
        log(f'PDF aberto. Total de páginas: {total_paginas}')
        return [p.extract_text().strip() if p.extract_text() else '' for p in leitor.pages]

def _identificar_paginas_para_ocr(textos_por_pagina: list) -> list:
    """Identifica páginas que necessitam de OCR baseando-se no limite de caracteres."""
    paginas_para_ocr = [i for i, t in enumerate(textos_por_pagina) if len(t) < LIMITE_CHARS_PAGINA]
    digitais = len(textos_por_pagina) - len(paginas_para_ocr)
    log(f'Páginas digitais: {digitais} | Páginas para OCR: {len(paginas_para_ocr)}')
    return paginas_para_ocr

def _aplicar_ocr_nas_paginas(caminho_pdf: str, paginas_para_ocr: list, textos_por_pagina: list):
    """Converte as páginas necessárias para imagem e aplica OCR in-place."""
    if not paginas_para_ocr:
        return
        
    log(f'Iniciando OCR em {len(paginas_para_ocr)} página(s)...', 'OCR')
    imagens = convert_from_path(
        caminho_pdf,
        dpi=200,
        first_page=min(paginas_para_ocr) + 1,
        last_page=max(paginas_para_ocr) + 1
    )
    
    indice_imagem = {
        num_pag: imagens[i]
        for i, num_pag in enumerate(range(min(paginas_para_ocr), max(paginas_para_ocr) + 1))
        if num_pag in paginas_para_ocr
    }
    
    ocr_ok = ocr_falha = 0
    for num_pag in paginas_para_ocr:
        imagem = indice_imagem.get(num_pag)
        if imagem:
            texto_ocr = ocr_pagina(imagem, num_pag + 1)
            if texto_ocr:
                textos_por_pagina[num_pag] = texto_ocr
                ocr_ok += 1
            else:
                textos_por_pagina[num_pag] = f'[PÁGINA {num_pag + 1}: OCR não extraiu conteúdo legível]'
                ocr_falha += 1
    log(f'OCR concluído. Sucesso: {ocr_ok} | Falha: {ocr_falha}', 'OCR')

def _formatar_textos_paginas(textos_por_pagina: list) -> str:
    """Formata os textos extraídos com marcação de slide."""
    texto_total = ''
    for num_pagina, texto in enumerate(textos_por_pagina, start=1):
        if texto:
            texto_total += f'--- Slide {num_pagina} ---\n{texto}\n\n'
    return texto_total

def extrair_texto_pdf(caminho_pdf: str) -> str:
    """
    Extrai texto de um PDF com detecção automática de páginas escaneadas.
    Estratégia: PyPDF2 primeiro; OCR com Tesseract nas páginas insuficientes.
    """
    if not caminho_pdf:
        return 'Nenhum slide fornecido para esta aula.'
    if not os.path.exists(caminho_pdf):
        return f'AVISO: PDF não encontrado no caminho especificado: {caminho_pdf}'

    try:
        textos_por_pagina = _ler_texto_direto_pdf(caminho_pdf)
        paginas_para_ocr = _identificar_paginas_para_ocr(textos_por_pagina)
        _aplicar_ocr_nas_paginas(caminho_pdf, paginas_para_ocr, textos_por_pagina)
        
        texto_total = _formatar_textos_paginas(textos_por_pagina)

        if not texto_total.strip():
            return (
                'AVISO: Nenhum conteúdo foi extraído do PDF — '
                'nem via leitura direta nem via OCR. '
                'O resumo será gerado apenas com base na transcrição da aula.'
            )

        log(f'Extração concluída. Total de caracteres: {len(texto_total):,}', 'OK')
        return texto_total

    except Exception as e:
        log(f'Erro inesperado na extração do PDF: {type(e).__name__}: {e}', 'ERR')
        return (
            f'AVISO: Falha na extração do PDF ({type(e).__name__}: {e}). '
            'O resumo será gerado apenas com base na transcrição da aula.'
        )

def montar_payload(transcricao: str, slides: str, nomes_audios: list, duracao_segundos_total: float) -> str:
    cabecalho_audios = ",".join(nomes_audios)
    duracao_str = formatar_duracao(duracao_segundos_total)
    
    return (
        f'**AUDIOS_ORIGEM:**{cabecalho_audios}\n\n'
        f'**DURACAO_TOTAL_DA_AULA:**{duracao_str}\n\n'
        '**TRANSCRIÇÃO_DA_AULA_EM_TEXTO_BRUTO:**\n'
        f'{transcricao}\n\n'
        '**CONTEÚDO_DOS_SLIDES_EM_TEXTO:**\n'
        f'{slides}'
    )

def salvar_txt_drive(conteudo: str, nome_saida: str, pasta_destino: str) -> str:
    os.makedirs(pasta_destino, exist_ok=True)
    caminho_final = os.path.join(pasta_destino, f'{nome_saida}.txt')
    with open(caminho_final, 'w', encoding='utf-8') as f:
        f.write(conteudo)
    tamanho_kb = os.path.getsize(caminho_final) / 1024
    log(f'Arquivo salvo: {caminho_final} ({tamanho_kb:.1f} KB)', 'OK')
    return caminho_final

# ═════════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL (PIPELINE)
# ═════════════════════════════════════════════════════════════════════

class TranscriptionPipeline:
    def __init__(self, aulas, pasta_saida):
        self.aulas = aulas
        self.pasta_saida = pasta_saida
        self.separador_visual = '=' * 65
        self.relatorio_final = []

    def run(self):
        tempo_inicio_lote = time.time()
        self._print_header()

        for idx, aula in enumerate(self.aulas, start=1):
            self._process_aula(idx, aula)

        self._print_report(tempo_inicio_lote)

    def _print_header(self):
        print(self.separador_visual)
        print('  PIPELINE DE TRANSCRIÇÃO MÉDICA — v2.4 (CUDA + OCR + Duração + Priming Auto)')
        print(f'  Início: {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}')
        print(f'  Aulas no lote: {len(self.aulas)}')
        print(self.separador_visual)

    def _process_aula(self, idx, aula):
        nome = formatar_nome_saida(aula['nome_saida'])
        audios = aula['caminhos_audios']
        pdf = aula.get('caminho_pdf')
        prompt = aula.get('prompt_whisper', '').strip()

        print(f'\n{self.separador_visual}')
        print(f'  AULA {idx}/{len(self.aulas)}: {nome}')
        print(self.separador_visual)

        tempo_inicio_aula = time.time()
        status_aula = 'SUCESSO'
        motivo_falha = ''

        try:
            duracao_total, transcricao_completa = self._step_1_transcribe(audios, prompt)
            texto_slides = self._step_2_extract_pdf(pdf)
            self._step_3_save(nome, audios, duracao_total, transcricao_completa, texto_slides)

        except (FileNotFoundError, RuntimeError, Exception) as e:
            status_aula, motivo_falha = self._handle_error(e)
        finally:
            self._cleanup_temp_dir()

        self.relatorio_final.append({
            'nome': nome,
            'status': status_aula,
            'duracao': time.time() - tempo_inicio_aula,
            'motivo': motivo_falha,
        })

    def _step_1_transcribe(self, audios, prompt):
        log(f'PASSO 1/3 — Transcrição de áudio ({len(audios)} parte(s))', 'STEP')
        transcricao_completa = ''
        duracao_total = 0.0

        for i, caminho_audio in enumerate(audios, start=1):
            duracao_parte = obter_duracao_audio_segundos(caminho_audio)
            duracao_total += duracao_parte
            
            if duracao_parte > 0:
                log(f'Duração da parte {i}: {int(duracao_parte // 60)}m {int(duracao_parte % 60)}s')

            texto_parte = transcrever_audio(
                caminho_audio=caminho_audio,
                parte_num=i,
                total_partes=len(audios),
                prompt=prompt,
                config=WHISPER_CONFIG
            )
            transcricao_completa += texto_parte
            if i < len(audios):
                transcricao_completa += SEPARADOR

        log(f'Transcrição total: {len(transcricao_completa):,} caracteres.', 'OK')
        if duracao_total > 0:
            log(f'Duração total da aula ({len(audios)} parte(s)): {formatar_duracao(duracao_total)}', 'OK')
        else:
            log('Duração não determinada (ffprobe indisponível ou falhou).', 'WARN')
            
        return duracao_total, transcricao_completa

    def _step_2_extract_pdf(self, pdf):
        log('PASSO 2/3 — Extração de texto do PDF (com detecção OCR)', 'STEP')
        return extrair_texto_pdf(pdf)

    def _step_3_save(self, nome, audios, duracao_total, transcricao_completa, texto_slides):
        log('PASSO 3/3 — Montagem do payload e envio ao Drive', 'STEP')
        nomes_audios = [Path(c).name for c in audios]
        documento_final = montar_payload(transcricao_completa, texto_slides, nomes_audios, duracao_total)
        salvar_txt_drive(documento_final, nome, self.pasta_saida)

    def _handle_error(self, e):
        if isinstance(e, FileNotFoundError):
            log(f'Arquivo não encontrado: {e}', 'ERR')
            return 'FALHA', f'FileNotFoundError: {e}'
        elif isinstance(e, RuntimeError):
            log(f'Erro no Whisper: {e}', 'ERR')
            return 'FALHA', f'RuntimeError: {str(e)[:200]}'
        else:
            log(f'Erro inesperado: {type(e).__name__}: {e}', 'ERR')
            return 'FALHA', f'{type(e).__name__}: {str(e)[:200]}'

    def _cleanup_temp_dir(self):
        if os.path.exists(DIR_TEMP):
            shutil.rmtree(DIR_TEMP)

    def _print_report(self, tempo_inicio_lote):
        duracao_lote = time.time() - tempo_inicio_lote
        print(f'\n{self.separador_visual}')
        print('  RELATÓRIO FINAL DO LOTE')
        print(self.separador_visual)

        sucesso = sum(1 for r in self.relatorio_final if r['status'] == 'SUCESSO')
        falhas = sum(1 for r in self.relatorio_final if r['status'] == 'FALHA')

        for r in self.relatorio_final:
            icone = '✅' if r['status'] == 'SUCESSO' else '❌'
            mins, segs = int(r['duracao'] // 60), int(r['duracao'] % 60)
            print(f"  {icone} {r['nome']} ({mins}m {segs}s)")
            if r['motivo']:
                print(f"     └─ Motivo: {r['motivo']}")

        print()
        print(f'  Resultado: {sucesso} sucesso(s), {falhas} falha(s)')
        print(f'  Duração total do lote: {int(duracao_lote // 60)}m {int(duracao_lote % 60)}s')
        print(f'  Arquivos prontos em: {self.pasta_saida}')
        print(self.separador_visual)

        if sucesso > 0:
            print(f'\n✅ {sucesso} arquivo(s) depositado(s) na fila do Google Drive.')
            print('\n📡 Acionando processamento de resumos no Google Apps Script (Webhook)...')
            self._trigger_webhook()

        if falhas > 0:
            print(f'\n⚠️  {falhas} aula(s) falharam. Verifique os erros acima e re-execute somente as aulas com falha.')
            
    def _trigger_webhook(self):
        try:
            import requests
            WEBHOOK_URL = "https://script.google.com/macros/s/AKfycbwNaI5m7rKM-i35fQMt5jsSbdinSkpZFYNS3_g-8RSPamfFvp-6UJz-Xwso-7nEObbtmQ/exec"
            response = requests.post(WEBHOOK_URL, allow_redirects=True)
            if response.status_code == 200:
                resultado = response.json()
                if resultado.get('status') == 'sucesso':
                    print('✅ Sucesso! O Apps Script concluiu a geração do resumo e dos flashcards.')
                else:
                    print(f'⚠️ O Apps Script retornou um aviso: {resultado.get("mensagem")}')
            else:
                print(f'❌ Falha de comunicação. Código HTTP: {response.status_code}')
        except Exception as e:
            print(f'❌ Falha ao conectar ao Webhook: {e}')

# Inicia a execução do pipeline passando as variáveis configuradas na Célula 3
if __name__ == "__main__":
    pipeline = TranscriptionPipeline(aulas=aulas_para_processar, pasta_saida=PASTA_SAIDA_DRIVE)
    pipeline.run()


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 5 — ACIONAMENTO DO WEBHOOK                  ║
# ║  Aciona o Apps Script para gerar resumos e cards    ║
# ╚══════════════════════════════════════════════════════╝

import requests

WEBHOOK_URL = "https://script.google.com/macros/s/AKfycbwNaI5m7rKM-i35fQMt5jsSbdinSkpZFYNS3_g-8RSPamfFvp-6UJz-Xwso-7nEObbtmQ/exec"

print('📡 Acionando processamento de resumos no Google Apps Script...')
try:
    response = requests.post(WEBHOOK_URL, allow_redirects=True)
    if response.status_code == 200:
        resultado = response.json()
        if resultado.get('status') == 'sucesso':
            print('✅ Sucesso! O Apps Script concluiu a geração do resumo e dos flashcards.')
        else:
            print(f'⚠️ O Apps Script retornou um aviso: {resultado.get("mensagem")}')
    else:
        print(f'❌ Falha de comunicação. Código HTTP: {response.status_code}')
        print(response.text)
except Exception as e:
    print(f'❌ Falha ao conectar ao Webhook: {e}')